In [ ]:
## Evidence Contract Validation

Before vulnerability analysis begins, every source is wrapped in a
versioned Aegis Evidence Record. The contract captures authorization,
integrity, provenance, freshness, parser confidence, evidence conflicts,
and multidimensional trust.

AegisSec does not permit low-quality or unauthorized evidence to silently
flow into automated vulnerability decisions.

In [2]:
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.validation.evidence_schema_validator import EvidenceSchemaValidator

record_path = (
    PROJECT_ROOT
    / "data"
    / "sample_inputs"
    / "evidence"
    / "valid_requirements_evidence.json"
)

validator = EvidenceSchemaValidator(
    PROJECT_ROOT / "schemas" / "aegis_evidence_record.schema.json"
)

result = validator.validate_file(record_path)

print(json.dumps(result.to_dict(), indent=2))

{
  "schema_valid": true,
  "business_rules_valid": true,
  "status": "accepted",
  "errors": [],
  "warnings": []
}


In [ ]:
## Mission-Aware Asset Context Validation

A vulnerability cannot be prioritized responsibly using CVSS, EPSS, or
sector name alone.

AegisSec therefore validates an accountable Asset Context Record containing:

- mission function;
- exposure;
- operational criticality;
- public-service impact;
- patient-safety impact;
- mission-readiness impact;
- data sensitivity;
- resilience requirements;
- asset ownership;
- evidence provenance.

The sector label is explicitly prohibited from directly determining
priority. Only documented and evidence-supported consequences may affect
the final policy decision.

In [3]:
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.validation.asset_context_validator import AssetContextValidator

asset_path = (
    PROJECT_ROOT
    / "data"
    / "sample_inputs"
    / "assets"
    / "valid_healthcare_asset.json"
)

asset_validator = AssetContextValidator(
    PROJECT_ROOT
    / "schemas"
    / "aegis_asset_context.schema.json"
)

asset_result = asset_validator.validate_file(asset_path)

print(json.dumps(asset_result.to_dict(), indent=2))

{
  "schema_valid": true,
  "business_rules_valid": true,
  "status": "accepted_with_warnings",
  "errors": [],
  "warnings": [
    {
      "code": "SYNTHETIC_DEMO_CONTEXT",
      "message": "This asset is a controlled demonstration fixture and must not be presented as a live operational asset.",
      "field_path": "provenance.origin_type"
    }
  ]
}


In [ ]:
## Provenance-Aware Vulnerability Intelligence

AegisSec does not collapse CVSS, EPSS, KEV, exploitation evidence,
affected-version ranges, and remediation guidance into one unexplained score.

Each intelligence dimension retains:

- its original source;
- evidence reference;
- retrieval date;
- freshness;
- missing-data status;
- source authority;
- conflicts;
- transformation provenance.

Important decision rules:

- CVSS measures technical severity, not complete organizational risk.
- Missing EPSS remains unknown and is never converted to zero.
- Absence from CISA KEV is not proof that exploitation does not exist.
- KEV listing cannot coexist with a no-known-exploitation conclusion.
- High-severity source conflicts require quarantine or human review.

In [4]:
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.validation.vulnerability_intelligence_validator import (
    VulnerabilityIntelligenceValidator,
)

intelligence_path = (
    PROJECT_ROOT
    / "data"
    / "sample_inputs"
    / "intelligence"
    / "valid_log4shell_intelligence.json"
)

intelligence_validator = VulnerabilityIntelligenceValidator(
    PROJECT_ROOT
    / "schemas"
    / "aegis_vulnerability_intelligence.schema.json"
)

intelligence_result = intelligence_validator.validate_file(
    intelligence_path
)

print(json.dumps(intelligence_result.to_dict(), indent=2))

{
  "schema_valid": true,
  "business_rules_valid": true,
  "status": "accepted_with_warnings",
  "errors": [],
  "warnings": [
    {
      "code": "EPSS_MISSING",
      "message": "EPSS data is missing and must remain unknown. Human review or cautious policy handling is required.",
      "field_path": "epss.status"
    },
    {
      "code": "SYNTHETIC_INTELLIGENCE_FIXTURE",
      "message": "This vulnerability intelligence record is a controlled fixture and must not be presented as a live intelligence retrieval.",
      "field_path": "provenance.origin_type"
    }
  ]
}


In [ ]:
## Tamper-Evident Aegis Decision Record

Every AegisSec finding produces a versioned Decision Record that binds:

- validated source evidence;
- asset and mission context;
- vulnerability intelligence;
- component identity;
- affectedness;
- policy minimum action;
- optional ML advisory;
- decision arbitration;
- human disposition;
- cryptographic audit lineage.

The Decision Record prevents silent modification and preserves the exact
inputs, policy version, model status, uncertainty, review requirement and
final accountable disposition.

The machine-learning model is advisory only and cannot lower a
non-overridable policy floor.

In [5]:
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.domain.decision_hashing import (
    verify_decision_record_hash,
)
from src.validation.decision_record_validator import (
    DecisionRecordValidator,
)

decision_path = (
    PROJECT_ROOT
    / "data"
    / "sample_inputs"
    / "decisions"
    / "valid_log4shell_decision_record.json"
)

decision_validator = DecisionRecordValidator(
    schema_path=(
        PROJECT_ROOT
        / "schemas"
        / "aegis_decision_record.schema.json"
    ),
    project_root=PROJECT_ROOT,
)

decision_result = decision_validator.validate_file(
    decision_path
)

with decision_path.open("r", encoding="utf-8") as file:
    decision_record = json.load(file)

print("Decision ID:", decision_record["decision_id"])
print(
    "System priority:",
    decision_record["arbitration"]["final_system_priority"],
)
print(
    "System action:",
    decision_record["arbitration"]["final_action"],
)
print(
    "Human review required:",
    decision_record["arbitration"]["human_review_required"],
)
print(
    "Record hash verified:",
    verify_decision_record_hash(decision_record),
)
print()
print(json.dumps(decision_result.to_dict(), indent=2))

Decision ID: AEG-DEC-LOG4J-HOSP-001
System priority: CRITICAL
System action: ACT
Human review required: True
Record hash verified: True

{
  "schema_valid": true,
  "business_rules_valid": true,
  "status": "accepted_with_warnings",
  "errors": [],
  "warnings": [
    {
      "code": "INPUT_ACCEPTED_WITH_WARNINGS",
      "message": "Referenced asset_context contains 1 warning(s).",
      "field_path": "input_records.reference.0"
    },
    {
      "code": "INPUT_ACCEPTED_WITH_WARNINGS",
      "message": "Referenced vulnerability_intelligence contains 2 warning(s).",
      "field_path": "input_records.reference.1"
    },
    {
      "code": "INPUT_ACCEPTED_WITH_WARNINGS",
      "message": "Referenced evidence_record contains 1 warning(s).",
      "field_path": "input_records.reference.2"
    },
    {
      "code": "ML_ADVISORY_NOT_RUN",
      "message": "The ML advisory model has not been run. The Decision Record currently relies on policy.",
      "field_path": "ml_advisory.status"
   

In [6]:
## Strict Parsing and Missing-Value Governance

Raw vulnerability data frequently contains strings, blank cells,
Boolean-like values, malformed numbers, locale-dependent dates and
inconsistent missing-value markers.

AegisSec does not rely on implicit Python or pandas conversions.

Every raw field is classified as:

- **Parsed**: a valid typed value was produced;
- **Missing**: no value exists and no replacement was invented;
- **Invalid**: the supplied value is malformed or outside its permitted range.

Important invariants:

- `"False"` must parse as `False`, not Python truthiness `True`.
- Missing EPSS must remain `None`, not `0`.
- A real EPSS score of `0` remains a valid observed value.
- `NaN`, infinity and out-of-range values are rejected.
- Dates must use unambiguous ISO formats.
- Invalid rows do not enter policy or machine-learning pipelines.

SyntaxError: invalid syntax (2019326143.py, line 3)

In [7]:
import csv
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.ingestion.vulnerability_feature_parser import (
    parse_vulnerability_feature_row,
)

parsing_fixture_path = (
    PROJECT_ROOT
    / "data"
    / "sample_inputs"
    / "parsing"
    / "strict_parsing_cases.csv"
)

parsing_demo_rows = []

with parsing_fixture_path.open(
    "r",
    encoding="utf-8",
    newline="",
) as file:
    reader = csv.DictReader(file)

    for raw_row in reader:
        report = parse_vulnerability_feature_row(raw_row)

        parsing_demo_rows.append(
            {
                "finding_id": raw_row["finding_id"],
                "raw_kev_flag": raw_row["kev_flag"],
                "parsed_kev_flag": report.values["kev_flag"],
                "raw_epss": raw_row["epss_probability"],
                "parsed_epss": report.values["epss_probability"],
                "epss_status": (
                    report.outcomes["epss_probability"]
                    .status.value
                ),
                "accepted": report.accepted,
                "error_codes": ", ".join(
                    error.code
                    for error in report.errors
                ) or "None",
                "warning_count": len(report.warnings),
            }
        )

strict_parsing_demo_df = pd.DataFrame(parsing_demo_rows)

display(strict_parsing_demo_df)

,finding_id,raw_kev_flag,parsed_kev_flag,raw_epss,parsed_epss,epss_status,accepted,error_codes,warning_count
0,FND-STRICT-001,False,False,,NaN,missing,True,None,1
1,FND-STRICT-002,0,False,0,0.0,parsed,True,None,3
2,FND-STRICT-003,maybe,None,NaN,NaN,invalid,False,"UNRECOGNIZED_BOOLEAN_TOKEN, NON_FINITE_NUMERIC...",0


In [ ]:
## Multidimensional Evidence Trust

AegisSec does not use a single subjective trust label.

Evidence is evaluated across eight independently explainable dimensions:

1. authenticity;
2. integrity;
3. completeness;
4. freshness;
5. consistency;
6. source authority;
7. parser confidence;
8. component-identity confidence.

The overall score uses a weighted geometric mean, but the aggregate score
cannot override mandatory trust floors.

Examples:

- unauthorized evidence is rejected;
- failed integrity is rejected;
- an unresolved high-severity conflict is quarantined;
- stale evidence is quarantined;
- synthetic evidence is clearly disclosed;
- a low parser-confidence score cannot be hidden by high scores elsewhere.

This implements the principle that AegisSec must never produce
high-confidence automated recommendations from low-trust evidence.

In [8]:
import copy
import json
import sys
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.governance.evidence_trust_engine import (
    EvidenceTrustEngine,
)

evidence_path = (
    PROJECT_ROOT
    / "data"
    / "sample_inputs"
    / "evidence"
    / "valid_requirements_evidence.json"
)

with evidence_path.open("r", encoding="utf-8") as file:
    base_evidence_record = json.load(file)

trust_engine = EvidenceTrustEngine(
    PROJECT_ROOT
    / "policies"
    / "trust"
    / "evidence_trust_policy_v1.yaml"
)

fixed_assessment_time = datetime(
    2026,
    7,
    13,
    18,
    45,
    tzinfo=timezone.utc,
)

high_trust_record = copy.deepcopy(base_evidence_record)
high_trust_record["evidence_id"] = "AEG-EVD-TRUST-HIGH-001"

stale_record = copy.deepcopy(base_evidence_record)
stale_record["evidence_id"] = "AEG-EVD-TRUST-STALE-001"
stale_record["freshness"]["status"] = "stale"
stale_record["freshness"]["age_seconds"] = 172800
stale_record["freshness"]["maximum_age_seconds"] = 86400

unauthorized_record = copy.deepcopy(base_evidence_record)
unauthorized_record["evidence_id"] = (
    "AEG-EVD-TRUST-UNAUTHORIZED-001"
)
unauthorized_record["authorization"]["status"] = "unauthorized"

trust_scenarios = {
    "High-trust controlled evidence": high_trust_record,
    "Stale evidence": stale_record,
    "Unauthorized evidence": unauthorized_record,
}

trust_summary_rows = []
trust_assessments = {}

for scenario_name, scenario_record in trust_scenarios.items():
    assessment = trust_engine.assess(
        scenario_record,
        assessed_at=fixed_assessment_time,
    )

    trust_assessments[scenario_name] = assessment

    trust_summary_rows.append(
        {
            "scenario": scenario_name,
            "aggregate_score": assessment.aggregate_score,
            "trust_level": assessment.trust_level,
            "action": assessment.action,
            "gates": ", ".join(
                gate.code
                for gate in assessment.gate_results
            ) or "None",
            "warnings": ", ".join(
                warning.code
                for warning in assessment.warnings
            ) or "None",
        }
    )

trust_summary_df = pd.DataFrame(trust_summary_rows)

display(trust_summary_df)

,scenario,aggregate_score,trust_level,action,gates,warnings
0,High-trust controlled evidence,0.9774,high,ACCEPT_WITH_WARNINGS,None,"SIGNATURE_NOT_VERIFIED, DECLARED_TRUST_MISMATCH"
1,Stale evidence,0.8276,low,QUARANTINE,"STALE_OR_EXPIRED_EVIDENCE, TRUST_DIMENSION_BEL...","SIGNATURE_NOT_VERIFIED, DECLARED_TRUST_MISMATCH"
2,Unauthorized evidence,0.8652,rejected,REJECT,"UNAUTHORIZED_EVIDENCE, TRUST_DIMENSION_BELOW_F...","SIGNATURE_NOT_VERIFIED, DECLARED_TRUST_MISMATCH"


In [9]:
high_trust_assessment = trust_assessments[
    "High-trust controlled evidence"
]

dimension_rows = []

for dimension_name, dimension in (
    high_trust_assessment.dimensions.items()
):
    dimension_rows.append(
        {
            "dimension": dimension_name,
            "score": dimension.score,
            "weight": dimension.weight,
            "mandatory_floor": dimension.minimum_score,
            "below_floor": dimension.below_floor,
            "reason_codes": ", ".join(
                dimension.reason_codes
            ),
        }
    )

trust_dimensions_df = pd.DataFrame(dimension_rows)

display(trust_dimensions_df)

,dimension,score,weight,mandatory_floor,below_floor,reason_codes
0,authenticity,0.9375,0.16,0.50,False,"AUTHORIZATION_AUTHORIZED, COLLECTION_BASIS_APP..."
1,integrity,1.0000,0.18,0.60,False,"INTEGRITY_VERIFIED, CRYPTOGRAPHIC_HASH_PRESENT..."
2,completeness,1.0000,0.12,0.50,False,"COMPONENT_VERSION_COVERAGE_EVALUATED, PURL_NOT..."
3,freshness,1.0000,0.12,0.50,False,"FRESHNESS_CURRENT, AGE_WITHIN_HALF_FRESHNESS_W..."
4,consistency,1.0000,0.14,0.50,False,NO_CONFLICTS_DETECTED
5,source_authority,0.9000,0.10,0.45,False,SOURCE_AUTHORITY_HIGH
6,parser_confidence,0.9800,0.10,0.60,False,PARSER_STATUS_SUCCESS
7,identity_confidence,1.0000,0.08,0.55,False,"VERSION_IDENTITY_COVERAGE_EVALUATED, NAME_VERS..."
